In [93]:
import geopandas as gpd
import pandas as pd 
import sys
from skimage import measure
from osgeo import gdal
import numpy as np
import logging
logging.getLogger("pyogrio").setLevel(logging.WARNING)

origin = '/workspace/'
sys.path.append('/media/')

thuenen_path = f"{origin}fields/09_Thuenen_field_maps/"

In [94]:
dat = pd.read_csv(f"{thuenen_path}TEMP/unique_pairs.csv")
# convert data types
dat['IACS'] = dat['IACS'].astype(str)
dat['HU BERLIN'] = dat['HU BERLIN'].astype(str)

# get field sizes
IACS_field_sizes = dat.groupby('IACS')['count'].sum()
HU_field_sizes = dat.groupby('HU BERLIN')['count'].sum()
# get polygons with maximum overlap
IACS_max_interesect = dat.loc[dat.groupby('IACS')['count'].idxmax()]
# HU_max_interesect = dat.loc[dat.groupby('HU BERLIN')['count'].idxmax()]

# merge field sizes to df
IACS_max_interesect = IACS_max_interesect.merge(IACS_field_sizes, on='IACS', how='left')
IACS_max_interesect = IACS_max_interesect.rename(columns={
    'count_x': 'count_intersect',
    'count_y': 'count_IACS_size'
})

IACS_max_interesect = IACS_max_interesect.merge(HU_field_sizes, on='HU BERLIN', how='left')
IACS_max_interesect = IACS_max_interesect.rename(columns={
    'count': 'count_HU_size'
})

In [95]:
# compute union and IoU
IACS_max_interesect['union'] = IACS_max_interesect['count_IACS_size'] + IACS_max_interesect['count_HU_size'] - IACS_max_interesect['count_intersect']
IACS_max_interesect['IoU'] = IACS_max_interesect['count_intersect'] / IACS_max_interesect['union']

In [96]:
# compute proportion of intersect to both polygons
IACS_max_interesect['rel_IACS_overlap'] = IACS_max_interesect['count_intersect'] / IACS_max_interesect['count_IACS_size']
IACS_max_interesect['rel_HU_overlap'] = IACS_max_interesect['count_intersect'] / IACS_max_interesect['count_HU_size']


In [97]:
# filter intersections with background (val=0) and fields < 5 Pixel
mask = IACS_max_interesect['IACS'].eq('0') | IACS_max_interesect['HU BERLIN'].eq('0') | \
    IACS_max_interesect['count_IACS_size'].le(5) | IACS_max_interesect['count_HU_size'].le(5)
IACS_max_interesect = IACS_max_interesect[~mask].copy()

In [98]:
for t1 in range(0,6,1):
    for t2 in range(0,6,1):
        t_hu = t1/10
        t_iacs = t2/10
        res_mask = IACS_max_interesect['rel_HU_overlap'].gt(t_hu) & IACS_max_interesect['rel_IACS_overlap'].gt(t_iacs)
        print(IACS_max_interesect[res_mask]['IoU'].mean())

0.5807789600133803
0.5807853008120444
0.5810997693511396
0.5828290384573362
0.5872718734162193
0.5948725949993744
0.6323069008047021
0.6323145436545953
0.6326426196926148
0.6345928170057644
0.6395503035736456
0.6478393819334805
0.6666520441260396
0.6666607236528158
0.6670248450762281
0.6692205411772301
0.6747825633232911
0.6838069989287128
0.6953445269453974
0.695354235231581
0.6957606691607099
0.6981683952123068
0.7043036873709176
0.7140066856524765
0.7201765147970961
0.720187301600768
0.7206276475180197
0.7232662595735273
0.7298378046427355
0.7402106373397148
0.7429443853705691
0.7429564132167443
0.743423939708693
0.746252518908305
0.7532156351098873
0.764242665196135
